# Study 2: Real-world analysis

In [45]:
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

import scipy.stats as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf

import plotly.express as px
import plotly.graph_objects as go

from utils.utils import *
from utils.variables import *

## Read in data

In [46]:
# Get CTRS data to show classifier performance on experimental data
df_pids, df_ctrs, df_general, df_comparisons = retrieve_analysed_data('clinicians')

print('----------------------------------------------------------------')

# Get real world data
df_rw_pids, df_rw_ctrs, df_rw_clinical, df_rw_human_labels, df_rw_feedback = retrieve_realworld_data()

df_rw_pids['pid'] = df_rw_pids['pid'].astype(str)
df_rw_clinical['pid'] = df_rw_clinical['pid'].astype(str)

Removing 7 transcripts that are not appropriate for CBT
N = 227


condition        model 
cognitive_layer  claude    25
                 gemini    26
                 gpt4      26
                 llama     24
human_therapist  human     26
standalone_llms  claude    25
                 gemini    24
                 gpt4      27
                 llama     24
dtype: int64

----------------------------------------------------------------
N users = 16630
N transcripts = 51916
N transcripts with autograded CTRS scores = 19674
N users with clinical outcomes = 942
N transcripts with human labels = 128
N transcripts with feedback = 6553


## CTRS Classifier

### Validation against experimental data

In [51]:
df = df_ctrs.copy()
df['human_score'] = df[['human1_score', 'human2_score']].apply(lambda x: x.dropna().iloc[0] if x.isna().sum() == 1 else x.mean(), axis=1)

df = (
    df
    .groupby('pid')[['human_score','classifier_score']]
    .mean()
    .reset_index()
    .dropna()
    .reset_index(drop=True)
)

print(f"N = {len(df)}")

compute_all_icc(df[['human_score','classifier_score']])

stat = sp.spearmanr(df[['human_score','classifier_score']])
print(f"Spearman's rho = {stat.correlation:.2f}, p = {readable_pvalue(stat.pvalue)}")

N = 227
Intraclass Correlation Coefficients (ICC) between human and classifier:
ICC(1,1): 0.464
ICC(1,k): 0.634
ICC(2,1): 0.519
ICC(2,k): 0.683
ICC(3,1): 0.654
ICC(3,k): 0.791
Spearman's rho = 0.71, p = 1.35e-35***


### Relationship to cognitive layer activation

Including curve-fitting procedure.

In [48]:
df = df_rw_pids.merge(df_rw_ctrs.drop(columns=['pid']),on='conversation_id',how='left')

mask = (df['n_user_messages']>=5) & (df['n_user_messages']<=30) & (df['n_characters']<5000)
df = df.loc[mask,].copy().reset_index(drop=True)

print(f"N users = {df['pid'].nunique()}")
print(f"N conversations = {df['conversation_id'].nunique()}")

df['z_ctrs_overall'] = zscore(df['ctrs_overall'])

# ------------------------------------------------------------------------------------------------------------------------------------
# Curve-fitting
# ------------------------------------------------------------------------------------------------------------------------------------

model_df = df.copy()

# Linear model
model_df['cl_transformed'] = model_df['cognitive_layer_score'].copy()
model_linear = smf.ols('z_ctrs_overall ~ cl_transformed + n_user_messages', data=model_df).fit()

# Log model
model_df['cl_transformed'] = model_df['cognitive_layer_score'].copy().apply(lambda x: np.log(x+1)   )
model_log = smf.ols('z_ctrs_overall ~ cl_transformed + n_user_messages', data=model_df).fit()

# Square root model
model_df['cl_transformed'] = model_df['cognitive_layer_score'].copy().apply(lambda x: np.sqrt(x))
model_sqrt = smf.ols('z_ctrs_overall ~ cl_transformed + n_user_messages', data=model_df).fit()

# Arctan model
model_df['cl_transformed'] = model_df['cognitive_layer_score'].copy().apply(lambda x: np.arctan(x))
model_arctan = smf.ols('z_ctrs_overall ~ cl_transformed + n_user_messages', data=model_df).fit()

# Hyperbolic model
model_df['cl_transformed'] = np.tanh(model_df['cognitive_layer_score']/model_df['cognitive_layer_score'].std())
model_hyperbolic = smf.ols('z_ctrs_overall ~ cl_transformed + n_user_messages', data=model_df).fit()

# Model comparison
model_comparison = pd.DataFrame({
    'model': ['linear', 'log', 'sqrt', 'arctan', 'hyperbolic'],
    'aic': [model_linear.aic, model_log.aic, model_sqrt.aic, model_arctan.aic, model_hyperbolic.aic],
    'bic': [model_linear.bic, model_log.bic, model_sqrt.bic, model_arctan.bic, model_hyperbolic.bic],
}).sort_values(by='bic',ascending=True)
model_comparison['bic_delta_next'] = model_comparison['bic'].diff()
model_comparison['bic_delta_best'] = model_comparison['bic'] - model_comparison['bic'].min()

print('Curve fitting model comparison:')
display(model_comparison)

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

model_df = df.copy()
model_df['z_cl'] = zscore(model_df['cognitive_layer_score'].apply(lambda x: np.sqrt(x)))

model = smf.ols('z_ctrs_overall ~ z_cl + n_user_messages', data=model_df).fit()
print_title('Sqrt cognitive layer score vs classified ctrs score')

print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")

# ------------------------------------------------------------------------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------------------------------------------------------------------------

fig = px.scatter(
    df,
    x='cognitive_layer_score',
    y='z_ctrs_overall',
    title=f'Cognitive Layer Activation vs CTRS',
    width=400,
    height=400,
    opacity=0.1,
    template='simple_white',
    labels={'z_ctrs_overall': 'Scaled CTRS', 'cognitive_layer_score': 'Cognitive Layer Activation'},
    color_discrete_sequence=['#EF5DA8'],
)
fig.update_layout(showlegend=False, coloraxis_showscale=False, font={'family': 'Arial'})
fig.update_xaxes(range=[2,41])

model_df = df.copy()
model_df['sqrt_cl'] = np.sqrt(model_df['cognitive_layer_score'])
model_sqrt = smf.ols('z_ctrs_overall ~ sqrt_cl + n_user_messages', data=model_df).fit()

curve_df = pd.DataFrame({
    'cognitive_layer_score': [x for x in range(2,41)],
    'n_user_messages': [model_df['n_user_messages'].mean()] * 39
    })
curve_df['sqrt_cl'] = np.sqrt(curve_df['cognitive_layer_score'])
curve_df['z_ctrs_overall'] = model_sqrt.predict(curve_df[['sqrt_cl','n_user_messages']])

fig.add_trace(
    go.Scatter(
        x=curve_df['cognitive_layer_score'].values,
        y=curve_df['z_ctrs_overall'].values,
        mode='lines',
        line=dict(shape='spline', color='#000000', width=2),
        showlegend=False
    )
)

fig.show()

fig.write_image('../results/real_world_ctrs_cl_activation.svg',width=400,height=400)

N users = 8920
N conversations = 19674
Curve fitting model comparison:


,model,aic,bic,bic_delta_next,bic_delta_best
2,sqrt,52282.403314,52306.064474,NaN,0.000000
1,log,52330.184988,52353.846147,47.781674,47.781674
0,linear,52591.676127,52615.337286,261.491139,309.272813
3,arctan,53703.819830,53727.480990,1112.143704,1421.416516
4,hyperbolic,54474.124956,54497.786116,770.305126,2191.721642


SQRT COGNITIVE LAYER SCORE VS CLASSIFIED CTRS SCORE
β = 0.66, 95% CI = [0.64, 0.68], P = 0.00e+00***


### Supplementary: Controlling for initial state & conversation complexity

In [49]:
df = df_rw_pids.merge(df_rw_ctrs.drop(columns=['pid']),on='conversation_id',how='left')

mask = (df['n_user_messages']>=5) & (df['n_user_messages']<=30) & (df['n_characters']<5000)
df = df.loc[mask,].copy().reset_index(drop=True)

print(f"N users = {df['pid'].nunique()}")
print(f"N conversations = {df['conversation_id'].nunique()}")

df['z_ctrs_overall'] = zscore(df['ctrs_overall'])

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

control_vars = [
    'n_words_first_message',
    'n_sentences_first_message',
    'mean_words_first_3_messages',
    'mean_sentences_first_3_messages',
    'dictionary_complexity',
    'embedding_entropy',
    'sentiment_polarity'
]
control_var_string = ' + '.join(control_vars)

model_df = df.copy()
model_df['z_cl'] = zscore(model_df['cognitive_layer_score'].apply(lambda x: np.sqrt(x)))

model = smf.ols('z_ctrs_overall ~ z_cl + n_user_messages + ' + control_var_string, data=model_df).fit()
print_title('Sqrt cognitive layer score vs classified ctrs score')
print(model.summary())

print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")

N users = 8920
N conversations = 19674
SQRT COGNITIVE LAYER SCORE VS CLASSIFIED CTRS SCORE
                            OLS Regression Results                            
Dep. Variable:         z_ctrs_overall   R-squared:                       0.221
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     620.5
Date:                Mon, 22 Sep 2025   Prob (F-statistic):               0.00
Time:                        12:17:17   Log-Likelihood:                -25457.
No. Observations:               19674   AIC:                         5.093e+04
Df Residuals:                   19664   BIC:                         5.101e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
-------------------

### Supplementary: Validation against human labels

In [ ]:
df = (
    df_rw_human_labels
    .copy()
    .groupby('conversation_id')[['human_score','classifier_score']]
    .mean()
    .reset_index()
    .melt(id_vars='conversation_id',value_vars=['human_score','classifier_score'],var_name='rater',value_name='score')
    )
df['rater'] = df['rater'].apply(lambda x: x.split('_')[0])

df = df.merge(df_rw_pids[['conversation_id','cognitive_layer_score','n_user_messages']],on='conversation_id',how='left')

print(f"N transcripts = {df.loc[df['rater']=='human','conversation_id'].nunique()}")

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

model_df = df.loc[df['rater'] == 'human',].copy().reset_index(drop=True)

model_df['z_ctrs_overall'] = zscore(model_df['score'])
model_df['z_cl'] = zscore(model_df['cognitive_layer_score'])

model = smf.ols('z_ctrs_overall ~ z_cl + n_user_messages', data=model_df).fit()
print(model.summary())

print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")

N transcripts = 128
Intraclass Correlation Coefficients (ICC) between human and classifier:
ICC(1,1): 0.321
ICC(1,k): 0.486
ICC(2,1): 0.342
ICC(2,k): 0.509
ICC(3,1): 0.364
ICC(3,k): 0.534
Spearman's rho = 0.45, p = 1.24e-07***
                            OLS Regression Results                            
Dep. Variable:         z_ctrs_overall   R-squared:                       0.204
Model:                            OLS   Adj. R-squared:                  0.191
Method:                 Least Squares   F-statistic:                     16.00
Date:                Mon, 22 Sep 2025   Prob (F-statistic):           6.52e-07
Time:                        12:17:23   Log-Likelihood:                -166.54
No. Observations:                 128   AIC:                             339.1
Df Residuals:                     125   BIC:                             347.6
Df Model:                           2                                         
Covariance Type:            nonrobust                         

## Clinical outcomes

### Cognitive layer score prediction

In [34]:
# Read in and filter data
df = df_rw_clinical.merge(
    df_rw_pids.groupby('pid')[['cognitive_layer_score','n_user_messages']].sum().reset_index(),on='pid',how='inner'
)

mask = (df['access_time_days']>=14)
n = df['pid'].nunique()
print(f"N participants = {n}")
n_exclude = df.loc[~mask,'pid'].nunique()
print(f"Excluding {n_exclude} participants who have fewer than 2 weeks between symptom measurements")

df = df.loc[mask,].copy().reset_index(drop=True)
print(f"N participants = {df['pid'].nunique()}")
print(f"--- NHS patients = {df.loc[df['source']=='nhs','pid'].nunique()}")
print(f"--- User research participants = {df.loc[df['source']=='user_research','pid'].nunique()}")

print(f'------ User research participants with 10+ weeks of data: {df.loc[(df["source"]=="user_research") & (df["access_time_days"]>=10*7),"pid"].nunique()}')

# Group by participant
df = df.groupby('pid').agg({
    'gad7_start': 'first',
    'phq9_start': 'first',
    'gad7_delta': 'first',
    'phq9_delta': 'first',
    'recovery': 'first',
    'cognitive_layer_score': 'sum',
    'n_user_messages': 'sum',
    'access_time_days': 'first',
    }).reset_index()

df['z_cl'] = zscore(np.sqrt(df['cognitive_layer_score']))
df['z_n_user_messages'] = zscore(df['n_user_messages']**2)

print(f"\nMedian time between symptom measurements = {df['access_time_days'].median():.2f} days ({df['access_time_days'].median()/7:.1f} weeks +/ {df['access_time_days'].std()/7:.1f})")

# Engagement metrics
engagement_df = (
    df_rw_pids
    .loc[df_rw_pids['pid'].isin(list(df['pid'].unique()))]
    .copy()
    .reset_index(drop=True)
)
engagement_df['n_exchanges'] = (engagement_df['n_user_messages'] * 2) + 1 # agent responds to each user message

n_conversations = engagement_df.groupby('pid')['conversation_id'].nunique().agg(['mean','std'])
print(f"N conversations per participant: M = {n_conversations['mean']:.1f}, SD = {n_conversations['std']:.1f}")

n_exchanges = engagement_df['n_exchanges'].agg(['mean','std'])
print(f"N exchanges per conversation: M = {n_exchanges['mean']:.1f}, SD = {n_exchanges['std']:.1f}, median = {engagement_df['n_exchanges'].median():.1f}")

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

# Symptom reduction
for outcome in ['gad7_delta','phq9_delta']:

    print_title(outcome)

    model = smf.ols(f'{outcome} ~ z_cl + z_n_user_messages + gad7_start + phq9_start', data=df).fit()
    print(model.summary())
    print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")
    display_residuals(model)

# Recovery rate
model_df = df.copy()
model_df['recovery'] = model_df['recovery'].astype(int)

model = smf.logit(f"recovery ~ z_cl + z_n_user_messages + gad7_start + phq9_start", data=model_df).fit()
print(model.summary())
print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")
odds_ratio = np.exp(model.params['z_cl'])
ci_lower = np.exp(model.conf_int().loc['z_cl', 0])
ci_upper = np.exp(model.conf_int().loc['z_cl', 1])
print(f"OR = {odds_ratio:.2f}, 95% CI = [{ci_lower:.2f}, {ci_upper:.2f}]")

display_residuals(model,'logistic')

# ------------------------------------------------------------------------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------------------------------------------------------------------------
plot_df = df.copy()
plot_df['recovery'] = plot_df['recovery'].astype(int)

bands = {
    'low': [0,.25],
    'medium': [.25,.75],
    'high': [.75,1]
}

plot_df['cl_band'] = None
for label,band in bands.items():
    q1 = plot_df['cognitive_layer_score'].quantile(band[0])
    q2 = plot_df['cognitive_layer_score'].quantile(band[1])
    mask = (plot_df['cognitive_layer_score']>=q1) & (plot_df['cognitive_layer_score']<=q2)
    plot_df.loc[mask,'cl_band'] = label

summary = plot_df.groupby('cl_band').agg({
    'recovery': ['mean','count'],
    'cognitive_layer_score': 'mean'
}).reset_index()
summary.columns = ['cl_band','recovery','count','cognitive_layer_score']
summary['sem'] = np.sqrt(summary['recovery'] * (1 - summary['recovery']) / summary['count'])
summary = summary.sort_values(by='cognitive_layer_score')
print(summary)

fig = px.bar(
    summary,
    x='cl_band',
    y='recovery',
    error_y='sem',
    title=f"Clinical recovery",
    labels={'cl_band': 'Cognitive Layer Activation', 'recovery': 'Recovery Rate'},
    color='cl_band',
    width=400,
    height=400,
    template='simple_white',
    category_orders={'cl_band': ['low', 'medium', 'high']},
    color_discrete_map={'high': '#ff68a8', 'medium': '#FF96C3', 'low': '#ffbbd8'},
)
fig.update_yaxes(tickformat='.0%',range=[0,.6])
fig.update_layout(font=dict(family='Arial'))
fig.show()

fig.update_layout(
    showlegend=False
)
fig.write_image(f"../results/realworld_recovery.svg",width=400,height=400)

N participants = 928
Excluding 147 participants who have fewer than 2 weeks between symptom measurements
N participants = 781
--- NHS patients = 485
--- User research participants = 296
------ User research participants with 10+ weeks of data: 84

Median time between symptom measurements = 73.00 days (10.4 weeks +/ 14.7)
N conversations per participant: M = 4.3, SD = 7.4
N exchanges per conversation: M = 17.6, SD = 12.5, median = 15.0
GAD7 DELTA
                            OLS Regression Results                            
Dep. Variable:             gad7_delta   R-squared:                       0.160
Model:                            OLS   Adj. R-squared:                  0.155
Method:                 Least Squares   F-statistic:                     36.89
Date:                Mon, 22 Sep 2025   Prob (F-statistic):           2.91e-28
Time:                        12:08:56   Log-Likelihood:                -2344.8
No. Observations:                 781   AIC:                             470

Shapiro-Wilk: 0.996, p = 0.031*
Kolmogorov-Smirnov: 0.029, p = 0.52
PHQ9 DELTA
                            OLS Regression Results                            
Dep. Variable:             phq9_delta   R-squared:                       0.176
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     41.43
Date:                Mon, 22 Sep 2025   Prob (F-statistic):           1.68e-31
Time:                        12:08:56   Log-Likelihood:                -2415.7
No. Observations:                 781   AIC:                             4841.
Df Residuals:                     776   BIC:                             4865.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------

Shapiro-Wilk: 0.998, p = 0.61
Kolmogorov-Smirnov: 0.021, p = 0.87
Optimization terminated successfully.
         Current function value: 0.620754
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               recovery   No. Observations:                  781
Model:                          Logit   Df Residuals:                      776
Method:                           MLE   Df Model:                            4
Date:                Mon, 22 Sep 2025   Pseudo R-squ.:                 0.08147
Time:                        12:08:57   Log-Likelihood:                -484.81
converged:                       True   LL-Null:                       -527.81
Covariance Type:            nonrobust   LLR p-value:                 9.304e-18
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.5913

  cl_band  recovery  count  cognitive_layer_score       sem
1     low  0.328042    189               2.243386  0.034151
2  medium  0.388175    389              20.318766  0.024709
0    high  0.517241    203             131.487685  0.035072


### Supplementary: Controlling for initial state & topic complexity

In [35]:
# Read in and filter data
df = df_rw_clinical.merge(
    df_rw_pids.groupby('pid')[['cognitive_layer_score','n_user_messages']].sum().reset_index(),on='pid',how='inner'
)

mask = (df['access_time_days']>=14)
n_exclude = df.loc[~mask,'pid'].nunique()
df = df.loc[mask,].copy().reset_index(drop=True)

df['z_cl'] = zscore(np.sqrt(df['cognitive_layer_score']))
df['z_n_user_messages'] = zscore(df['n_user_messages']**2)

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

control_vars = [
    'n_words_first_message',
    'n_sentences_first_message',
    'mean_words_per_message',
    'mean_sentences_per_message',
    'dictionary_complexity',
    'embedding_entropy',
    'sentiment_polarity',
    'topic'
]

control_var_string = ' + '.join(control_vars)
control_var_string = control_var_string.replace('topic','C(topic)')

# Symptom reduction
for outcome in ['gad7_delta','phq9_delta']:

    print_title(outcome)

    model = smf.ols(f'{outcome} ~ z_cl + z_n_user_messages +gad7_start + phq9_start + {control_var_string}', data=df).fit()
    print(model.summary())
    print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")
    display_residuals(model)

# Recovery rate
model_df = df.copy()
model_df['recovery'] = model_df['recovery'].astype(int)

model = smf.logit(f"recovery ~ z_cl + z_n_user_messages + gad7_start + phq9_start + {control_var_string}", data=model_df).fit()
print(model.summary())
print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")
odds_ratio = np.exp(model.params['z_cl'])
ci_lower = np.exp(model.conf_int().loc['z_cl', 0])
ci_upper = np.exp(model.conf_int().loc['z_cl', 1])
print(f"OR = {odds_ratio:.2f}, 95% CI = [{ci_lower:.2f}, {ci_upper:.2f}]")

display_residuals(model,'logistic')

GAD7 DELTA
                            OLS Regression Results                            
Dep. Variable:             gad7_delta   R-squared:                       0.216
Model:                            OLS   Adj. R-squared:                  0.187
Method:                 Least Squares   F-statistic:                     7.485
Date:                Mon, 22 Sep 2025   Prob (F-statistic):           3.72e-23
Time:                        12:08:58   Log-Likelihood:                -2103.1
No. Observations:                 706   AIC:                             4258.
Df Residuals:                     680   BIC:                             4377.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

Shapiro-Wilk: 0.996, p = 0.12
Kolmogorov-Smirnov: 0.033, p = 0.41
PHQ9 DELTA
                            OLS Regression Results                            
Dep. Variable:             phq9_delta   R-squared:                       0.217
Model:                            OLS   Adj. R-squared:                  0.188
Method:                 Least Squares   F-statistic:                     7.539
Date:                Mon, 22 Sep 2025   Prob (F-statistic):           2.33e-23
Time:                        12:08:58   Log-Likelihood:                -2175.8
No. Observations:                 706   AIC:                             4404.
Df Residuals:                     680   BIC:                             4522.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------

Shapiro-Wilk: 0.999, p = 0.89
Kolmogorov-Smirnov: 0.018, p = 0.97
Optimization terminated successfully.
         Current function value: 0.605698
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               recovery   No. Observations:                  706
Model:                          Logit   Df Residuals:                      680
Method:                           MLE   Df Model:                           25
Date:                Mon, 22 Sep 2025   Pseudo R-squ.:                 0.09569
Time:                        12:08:58   Log-Likelihood:                -427.62
converged:                       True   LL-Null:                       -472.87
Covariance Type:            nonrobust   LLR p-value:                 2.375e-09
                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

## Feedback

User feedback per conversation

In [36]:
df = (
    df_rw_feedback
    .copy()
    .merge(df_rw_pids[['conversation_id','cognitive_layer_score','n_user_messages']],on='conversation_id',how='left')
)

print('Feedback per conversation:')
table = df['feedback'].value_counts().reset_index()
table.columns = ['feedback','count']
table['percent'] = table['count'] / table['count'].sum()
display(table)

# ------------------------------------------------------------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------------------------------------------------------------

model_df = df.copy()

model_df['feedback'] = (model_df['feedback']=='helpful').astype(int)

model_df['z_cl'] = zscore(np.sqrt(model_df['cognitive_layer_score']))
model_df['z_n_user_messages'] = zscore(model_df['n_user_messages']**2)

model = smf.logit('feedback ~ z_cl + z_n_user_messages', data=model_df).fit()
print(model.summary())
print(f"β = {model.params['z_cl']:.2f}, 95% CI = [{model.conf_int().loc['z_cl',0]:.2f}, {model.conf_int().loc['z_cl',1]:.2f}], P = {readable_pvalue(model.pvalues['z_cl'])}")
odds_ratio = np.exp(model.params['z_cl'])
ci_lower = np.exp(model.conf_int().loc['z_cl', 0])
ci_upper = np.exp(model.conf_int().loc['z_cl', 1])
print(f"OR = {odds_ratio:.2f}, 95% CI = [{ci_lower:.2f}, {ci_upper:.2f}]")

display_residuals(model,'logistic')

# ------------------------------------------------------------------------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------------------------------------------------------------------------

# Top histogram
fig = px.histogram(
    df,
    x='cognitive_layer_score',
    color='feedback',
    barmode='overlay',
    title='Feedback by Cognitive Layer Score',
    width=400,
    height=250,
    color_discrete_map={
        'helpful': px.colors.qualitative.Plotly[2],
        'not helpful':px.colors.qualitative.Plotly[1]
        },
    template='simple_white',
    # histnorm='percent',
    labels={
        'cognitive_layer_score': 'Cognitive Layer Activation',
        'feedback': 'Feedback'
        },
    nbins=50,
    opacity=1
)
fig.update_yaxes(title='Conversations')
fig.update_xaxes(range=[df['cognitive_layer_score'].min(),df['cognitive_layer_score'].max()])
fig.update_layout(font={'family': 'Arial'})
fig.show()

# Bottom scatter plot
plot_df = df.groupby(['cognitive_layer_score','feedback']).size().reset_index(name='count')
plot_df['n'] = plot_df.groupby('cognitive_layer_score')['count'].transform('sum')
plot_df['p_helpful'] = plot_df['count'] / plot_df['n']
plot_df = plot_df.loc[plot_df['feedback']=='helpful',].copy().reset_index(drop=True)
plot_df['size'] = plot_df['n'].apply(lambda x: np.log(x+1))

model_df = df.copy()
model_df['feedback'] = (model_df['feedback']=='helpful').astype(int)
model = smf.logit('feedback ~ cognitive_layer_score', data=model_df).fit()

plot_df['smoothed'] = model.predict(plot_df['cognitive_layer_score'])

fig = px.scatter(
    plot_df,
    x='cognitive_layer_score',
    y='p_helpful',
    size='size',
    width=400,
    height=250,
    template='simple_white',
    color_discrete_sequence=['#EF5DA8'],
    size_max=12,
    opacity=0.4
)
fig.update_xaxes(title='Cognitive Layer Activation',range=[df['cognitive_layer_score'].min(),df['cognitive_layer_score'].max()+2])
fig.update_yaxes(title='Positive Feedback',tickformat=".0%",range=[.77,1.03])
fig.update_layout(font={'family': 'Arial'})

fig.add_trace(
    go.Scatter(
        x=plot_df['cognitive_layer_score'],
        y=plot_df['smoothed'],
        mode='lines',
        line=dict(shape='spline', color='#EF5DA8', width=2),
        fill='tozeroy',
        showlegend=False
    )
)

fig.show()

Feedback per conversation:


,feedback,count,percent
0,helpful,5700,0.869831
1,not helpful,853,0.130169


Optimization terminated successfully.
         Current function value: 0.376825
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               feedback   No. Observations:                 6553
Model:                          Logit   Df Residuals:                     6550
Method:                           MLE   Df Model:                            2
Date:                Mon, 22 Sep 2025   Pseudo R-squ.:                 0.02556
Time:                        12:08:58   Log-Likelihood:                -2469.3
converged:                       True   LL-Null:                       -2534.1
Covariance Type:            nonrobust   LLR p-value:                 7.428e-29
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.9722      0.039     50.339      0.000       1.895       2.049
z_cl    

Optimization terminated successfully.
         Current function value: 0.378418
         Iterations 7
